# Output & Trace Rendering — Demo

Every line the engine prints goes through one seam in `text_adventure_games/reporting.py`:
the engine emits a typed **`Message`** tagged with a **`Channel`** (what *kind* of
information it is — world narration, a blocked action, an agent's private reasoning…),
and a **`Renderer`** decides how those messages *look* on one surface (a colored
terminal, the Flask web app, a test capture). **Swap the renderer, not the engine**, and
the same game prints to a terminal, buffers dicts for the web, or records structured
messages for a test.

This notebook is fully **offline and deterministic** (no API key). It covers:

1. The **channels** — the kinds of information.
2. A small bag of sample **messages** spanning every channel.
3. **Verbosity** — `QUIET` / `NORMAL` / `VERBOSE` and which channels each shows.
4. **Renderers** — the same messages on two surfaces: `PlainRenderer`, then
   `RichTerminalRenderer` in color.
5. **`CaptureRenderer`** — recording messages for tests / introspection.
6. The seam **wired into a real game**.

For a live agent trace rendered in color, see `02_agents_react.ipynb`. For a prose guide
to interpreting the terminal output, see `docs/reading-the-output.md`.

## 1. Channels — *what kind* of information

A `Channel` is the *meaning* of a message, independent of how any surface draws it. The
four `AGENT_*` channels are the agent's **private ReAct trace** — they never enter another
character's observations and are hidden at lower verbosities.

In [1]:
from text_adventure_games.reporting import Channel, AGENT_CHANNELS

for ch in Channel:
    tag = "  (agent-private)" if ch in AGENT_CHANNELS else ""
    print(f"{ch.name:<18} {ch.value}{tag}")

NARRATION          narration
NPC_NARRATION      npc_narration
BLOCKED            blocked
CONFLICT           conflict
COMMAND            command
AGENT_OBSERVATION  agent_observation  (agent-private)
AGENT_REASONING    agent_reasoning  (agent-private)
AGENT_ACTION       agent_action  (agent-private)
AGENT_REFLECTION   agent_reflection  (agent-private)
SYSTEM             system


## 2. A scene as messages

A `Message` carries its `channel` (the meaning), the raw `text`, and optionally the
`actor` it's about and the `turn` it happened on. Here is one short combat exchange
expressed as messages spanning every channel — we'll render it different ways below.

In [2]:
from text_adventure_games.reporting import Message, Channel

scene = [
    Message(Channel.SYSTEM, "Turn 1", turn=1),
    Message(Channel.COMMAND, "attack troll with sword", actor="player", turn=1),
    Message(Channel.NARRATION, "You swing your sword at the troll.", actor="player", turn=1),
    Message(Channel.AGENT_OBSERVATION, "DRAWBRIDGE\nA troll blocks the bridge.", actor="troll", turn=1),
    Message(Channel.AGENT_REASONING, "The player struck me; I should fight back.", actor="troll", turn=1),
    Message(Channel.AGENT_ACTION, "attack player with club", actor="troll", turn=1),
    Message(Channel.NPC_NARRATION, "The troll swings its club at you.", actor="troll", turn=1),
    Message(Channel.AGENT_REFLECTION, "I missed; next turn I'll aim for the legs.", actor="troll", turn=1),
    Message(Channel.CONFLICT, "guard got the fish first this turn.", actor="troll", turn=1),
]
print(f"{len(scene)} messages across {len({m.channel for m in scene})} channels")

9 messages across 9 channels


## 3. Verbosity — how much to show

A renderer's `level` gates which channels it shows. `channel_visible(channel, level)` is
the rule. The base channels (world narration, blocked, conflict, command, system) always
show; `NORMAL` adds the agent's reasoning/action/reflection; `VERBOSE` adds the full
observation too.

In [3]:
from text_adventure_games.reporting import channel_visible, QUIET, NORMAL, VERBOSE

levels = [QUIET, NORMAL, VERBOSE]
print(f"{'channel':<20}" + "".join(f"{lvl:<9}" for lvl in levels))
print("-" * (20 + 9 * len(levels)))
for ch in Channel:
    cells = "".join(f"{('show' if channel_visible(ch, lvl) else '·'):<9}" for lvl in levels)
    print(f"{ch.name:<20}{cells}")

channel             quiet    normal   verbose  
-----------------------------------------------
NARRATION           show     show     show     
NPC_NARRATION       show     show     show     
BLOCKED             show     show     show     
CONFLICT            show     show     show     
COMMAND             show     show     show     
AGENT_OBSERVATION   ·        ·        show     
AGENT_REASONING     ·        show     show     
AGENT_ACTION        ·        show     show     
AGENT_REFLECTION    ·        show     show     
SYSTEM              show     show     show     


## 4. Renderers — same messages, different surface

`PlainRenderer` is the color-free fallback (used when `rich` isn't installed, stdout
isn't a TTY, or `NO_COLOR` is set) — and what keeps test/CI output deterministic. Render
the same `scene` at each verbosity and watch the agent trace appear:

In [4]:
from text_adventure_games.reporting import PlainRenderer

for level in (QUIET, NORMAL, VERBOSE):
    print("=" * 60)
    print(f"PlainRenderer(level={level!r})")
    print("=" * 60)
    renderer = PlainRenderer(level=level)
    for m in scene:
        renderer.emit(m)
    print()

PlainRenderer(level='quiet')
Turn 1
> attack troll with sword
You swing your sword at the troll.
The troll swings its club at you.
⚔ guard got the fish first this turn.

PlainRenderer(level='normal')
Turn 1
> attack troll with sword
You swing your sword at the troll.
troll [reasoning] The player struck me; I should fight back.
troll [action] attack player with club
The troll swings its club at you.
troll [reflect] I missed; next turn I'll aim for the legs.
⚔ guard got the fish first this turn.

PlainRenderer(level='verbose')
Turn 1
> attack troll with sword
You swing your sword at the troll.
troll [observe]
DRAWBRIDGE
A troll blocks the bridge.
troll [reasoning] The player struck me; I should fight back.
troll [action] attack player with club
The troll swings its club at you.
troll [reflect] I missed; next turn I'll aim for the legs.
⚔ guard got the fish first this turn.



`RichTerminalRenderer` is the colored, turn-structured default in an interactive
terminal. It draws a rule at each turn and gives **every line a bracketed channel
label** — `[narration]`, `[action]`, `[observation]`, … — so the *kind* of line reads
from the text alone (color is only a secondary cue, which matters when several channels
would otherwise share a hue); agent-trace lines are also prefixed with the actor
(`troll [reasoning] …`). It's imported lazily, so the engine never *hard*-requires `rich`.

In a notebook, stdout isn't a TTY, so `default_renderer()` would fall back to plain text —
we pass a `Console(force_jupyter=True)` to force the colored output here. The web app
supplies its own `WebRenderer` (in `webapp/web_parser.py`) that speaks the template's
`{"type", "text"}` dicts — same messages, a different surface.

In [5]:
from rich.console import Console

from text_adventure_games.reporting import RichTerminalRenderer

# force_jupyter=True makes rich emit its colored HTML in the notebook (stdout
# here isn't a TTY, so default_renderer() would otherwise pick the plain fallback).
rich_renderer = RichTerminalRenderer(
    level=VERBOSE, console=Console(force_jupyter=True, width=88)
)
rich_renderer.turn_header(1, "8:45 AM")
for m in scene:
    rich_renderer.emit(m)

Turn 1 · 8:45 AM ───────────────────────────────────────────────────────────────────────

[system] Turn 1

[player command] attack troll with sword

[narration] You swing your sword at the troll.

troll [observation] DRAWBRIDGE
                    A troll blocks the bridge.

troll [reasoning] The player struck me; I should fight back.

troll [action] attack player with club

[npc] The troll swings its club at you.

troll [reflection] I missed; next turn I'll aim for the legs.

[conflict] guard got the fish first this turn.

## 5. `CaptureRenderer` — record instead of print

For tests and introspection, `CaptureRenderer` keeps the messages so you can assert on
**channels** (and actor/text), not on formatted bytes. It defaults to `VERBOSE`, so it
sees everything.

In [6]:
from text_adventure_games.reporting import CaptureRenderer

cap = CaptureRenderer()  # VERBOSE by default
for m in scene:
    cap.emit(m)

print("captured:", len(cap.messages), "messages")
print("reasoning:", cap.texts(Channel.AGENT_REASONING))
print("conflict: ", cap.texts(Channel.CONFLICT))

captured: 9 messages
reasoning: ['The player struck me; I should fight back.']
conflict:  ['guard got the fish first this turn.']


## 6. Wired into a real game

You don't build messages by hand — the parser emits them as actions resolve. Point a
game's parser at a renderer and play: here a `CaptureRenderer` records a successful pickup
(`NARRATION`) and a failed one (`BLOCKED`).

In [7]:
from collections import Counter

from text_adventure_games import games, things
from text_adventure_games.reporting import CaptureRenderer, Channel

room = things.Location("Room", "A plain room.")
player = things.Character("player", "the player", "I explore.")
game = games.Game(room, player, characters=[])
room.add_item(things.Item("lamp", "a brass lamp"))

cap = CaptureRenderer()
game.parser.set_renderer(cap)

game.parser.parse_command("get lamp", actor=player)       # succeeds -> NARRATION
game.parser.parse_command("get unicorn", actor=player)    # fails    -> BLOCKED

for name, n in Counter(m.channel.name for m in cap.messages).items():
    print(f"{name:<14} x{n}")
print()
print("NARRATION:", cap.texts(Channel.NARRATION))
print("BLOCKED:  ", cap.texts(Channel.BLOCKED))

NARRATION      x1
BLOCKED        x2

NARRATION: ['player got the lamp.']
BLOCKED:   ["I don't see it.", "I don't see it."]


### Takeaways

- The engine speaks in **`Message`s on `Channel`s**; renderers decide how they look.
- Set verbosity with `OUTPUT_LEVEL=quiet|normal|verbose`, or a renderer's `level=`.
- Swap surfaces with `game.parser.set_renderer(...)`: `RichTerminalRenderer` (color),
  `PlainRenderer` (plain/CI), `CaptureRenderer` (tests), `WebRenderer` (Flask).
- See `02_agents_react.ipynb` for a live colored agent trace and
  `docs/reading-the-output.md` for a reader's guide.